In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
from torchvision import models
import numpy as np
import time
# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
# =========================
# LABEL CORRUPTION
# =========================
def corrupt_labels(labels, ratio, num_classes=10):
    labels = labels.clone()
    n = len(labels)
    k = int(n * ratio)

    idx = torch.randperm(n)[:k]
    rand = torch.randint(0, num_classes, (k,))

    same = rand == labels[idx]
    while same.any():
        rand[same] = torch.randint(0, num_classes, (same.sum(),))
        same = rand == labels[idx]

    labels[idx] = rand
    return labels

class CorruptedDataset(Dataset):
    def __init__(self, dataset, corruption):
        self.dataset = dataset
        labels = torch.tensor(dataset.targets)
        self.labels = corrupt_labels(labels, corruption)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, _ = self.dataset[idx]
        return x, self.labels[idx]

# =========================
# AUGMENTATIONS (MoCo)
# =========================
class TwoCropsTransform:
    def __init__(self, base_transform):
        self.base_transform = base_transform

    def __call__(self, x):
        return [self.base_transform(x), self.base_transform(x)]

moco_transform = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.4,0.4,0.4,0.1),
    T.RandomGrayscale(p=0.2),
    T.ToTensor(),
    T.Normalize([0.4914,0.4822,0.4465],
                [0.247,0.243,0.261])
])

test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize([0.4914,0.4822,0.4465],
                [0.247,0.243,0.261])
])

# =========================
# RESNET CIFAR
# =========================
def get_resnet18_cifar():
    model = models.resnet18(pretrained=False)
    model.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
    model.maxpool = nn.Identity()
    return model

# =========================
# MoCo MODEL
# =========================
class MoCo(nn.Module):
    def __init__(self, dim=128, K=4096, m=0.999, T=0.07):
        super().__init__()

        self.K, self.m, self.T = K, m, T

        self.encoder_q = get_resnet18_cifar()
        self.encoder_k = get_resnet18_cifar()

        dim_mlp = self.encoder_q.fc.in_features

        self.encoder_q.fc = nn.Sequential(
            nn.Linear(dim_mlp, dim_mlp),
            nn.ReLU(),
            nn.Linear(dim_mlp, dim)
        )

        self.encoder_k.fc = nn.Sequential(
            nn.Linear(dim_mlp, dim_mlp),
            nn.ReLU(),
            nn.Linear(dim_mlp, dim)
        )

        for p_q, p_k in zip(self.encoder_q.parameters(), self.encoder_k.parameters()):
            p_k.data.copy_(p_q.data)
            p_k.requires_grad = False

        self.register_buffer("queue", F.normalize(torch.randn(dim, K), dim=0))
        self.register_buffer("queue_ptr", torch.zeros(1, dtype=torch.long))

    @torch.no_grad()
    def momentum_update(self):
        for p_q, p_k in zip(self.encoder_q.parameters(), self.encoder_k.parameters()):
            p_k.data = p_k.data * self.m + p_q.data * (1 - self.m)

    @torch.no_grad()
    def dequeue_enqueue(self, keys):
        bs = keys.shape[0]
        ptr = int(self.queue_ptr)

        self.queue[:, ptr:ptr+bs] = keys.T
        ptr = (ptr + bs) % self.K
        self.queue_ptr[0] = ptr

    def forward(self, im_q, im_k):
        q = F.normalize(self.encoder_q(im_q), dim=1)

        with torch.no_grad():
            self.momentum_update()
            k = F.normalize(self.encoder_k(im_k), dim=1)

        l_pos = torch.einsum('nc,nc->n', [q, k]).unsqueeze(-1)
        l_neg = torch.einsum('nc,ck->nk', [q, self.queue.clone().detach()])

        logits = torch.cat([l_pos, l_neg], dim=1) / self.T
        labels = torch.zeros(logits.size(0), dtype=torch.long).to(device)

        self.dequeue_enqueue(k)
        return logits, labels

# =========================
# TRAIN MoCo (Adam + Early Stopping)
# =========================
def train_moco(model, loader, max_epochs=200, patience=20, min_delta=1e-3):
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_loss = float("inf")
    patience_counter = 0

    for epoch in range(max_epochs):
        start = time.time()
        total_loss = 0

        for (x1, x2), _ in loader:
            x1 = x1.to(device, non_blocking=True)
            x2 = x2.to(device, non_blocking=True)

            logits, labels = model(x1, x2)
            loss = F.cross_entropy(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(loader)
        end = time.time()
        print(f"[MoCo] Epoch {epoch}, Loss: {avg_loss:.4f}, Time: {end-start:.2f}s")

        # Early stopping
        if best_loss - avg_loss > min_delta:
            best_loss = avg_loss
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"MoCo converged at epoch {epoch}")
            break

# =========================
# LINEAR PROBE
# =========================
def train_linear(encoder, classifier, loader, epochs=50):
    optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)

    for epoch in range(epochs):
        total_loss = 0

        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with torch.no_grad():
                f = encoder(x)

            out = classifier(f)
            loss = F.cross_entropy(out, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"[Linear] Epoch {epoch}, Loss: {total_loss/len(loader):.4f}")

def evaluate(encoder, classifier, loader):
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            f = encoder(x)
            out = classifier(f)
            pred = out.argmax(1)

            correct += (pred == y).sum().item()
            total += y.size(0)

    acc = 100 * correct / total
    print(f"Accuracy: {acc:.2f}%")
    return acc

# =========================
# DATA
# =========================
ssl_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True,
    transform=TwoCropsTransform(moco_transform)
)

ssl_loader = DataLoader(ssl_dataset, batch_size=256, shuffle=True, drop_last=True, num_workers=4, pin_memory= True)

train_eval_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True,
    transform=test_transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True,
    transform=test_transform
)

test_loader = DataLoader(test_dataset, batch_size=256, num_workers=4, pin_memory=True)



In [ ]:
# =========================
# MAIN EXPERIMENT (STRICT VICREG MATCH)
# =========================
corruptions = [0.0, 0.1, 0.3, 0.5]
results = []

for c in corruptions:
    print(f"\n===== CORRUPTION {c*100}% =====")

    # =========================
    # 1. Train MoCo
    # =========================
    moco = MoCo().to(device)
    train_moco(moco, ssl_loader)

    # =========================
    # 2. Extract encoder (REMOVE projection head)
    # =========================
    encoder = moco.encoder_q
    encoder.fc = nn.Identity()   # 🔥 CRUCIAL (same as VICReg 512-d)

    # =========================
    # 3. Corrupt labels
    # =========================
    corrupted_dataset = CorruptedDataset(train_eval_dataset, corruption=c)
    train_loader = DataLoader(corrupted_dataset, batch_size=256, shuffle=True, num_workers=4, pin_memory=True)

    # =========================
    # 4. Freeze encoder
    # =========================
    for p in encoder.parameters():
        p.requires_grad = False
    encoder.eval()

    # =========================
    # 5. Linear probe
    # =========================
    classifier = nn.Linear(512, 10).to(device)

    train_linear(encoder, classifier, train_loader, epochs=50)

    # =========================
    # 6. Evaluate
    # =========================
    acc = evaluate(encoder, classifier, test_loader)
    results.append((c, acc))


# =========================
# RESULTS
# =========================
print("\nFINAL RESULTS:")
for c, acc in results:
    print(f"Corruption {c*100}% -> {acc:.2f}%")